# LAB | RAG with the training wheels off

## Step 1: Setup and Document Preparation

In [1]:
pip install openai numpy python-dotenv pypdf tiktoken -q


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Key loaded: True


In [ ]:
# Load all source documents from the "input" folder
import os
from pypdf import PdfReader

def load_pdf(path: str) -> str:
    """Extract raw text from a single PDF file."""
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_documents_from_folder(folder: str) -> dict[str, str]:
    """
    Load every PDF in a folder.
    Returns a dict mapping filename -> raw text,
    so we can later trace a chunk back to its source document.
    """
    documents = {}
    for filename in os.listdir(folder):
        if filename.endswith(".pdf"):
            path = os.path.join(folder, filename)
            documents[filename] = load_pdf(path)
    return documents

docs = load_documents_from_folder("input")
for name, text in docs.items():
    print(f"{name}: {len(text)} characters")

ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf: 156411 characters
385082eng.pdf: 22103 characters


In [ ]:
# Split each document into overlapping chunks, keeping track of source
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping chunks by character count.
    Overlap prevents a relevant sentence from being cut in half
    right at a chunk boundary.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap  # move forward, keeping the overlap region
    return chunks

# all_chunks stores (source_document, chunk_text) tuples
all_chunks = []
for filename, text in docs.items():
    for chunk in chunk_text(text):
        all_chunks.append((filename, chunk))

print(f"Total chunks across all documents: {len(all_chunks)}")
print(f"Example chunk from '{all_chunks[0][0]}':")
print(all_chunks[0][1][:200])

Total chunks across all documents: 398
Example chunk from 'ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf':
INDEPENDENT 
HIGH-LEVEL EXPERT GROUP ON 
ARTIFICIAL INTELLIGENCE 
SET UP BY THE EUROPEAN COMMISSION 
 
 
 
 
 
 
ETHICS GUIDELINES 
FOR TRUSTWORTHY AI 
 
 
 
 
 
 
ETHICS GUIDELINES FOR TRUSTWORTHY AI


## Step 2: Generate Embeddings

In [ ]:
# Generate embeddings in batches instead of one-by-one
def get_embeddings_batch(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    """Call the OpenAI embeddings endpoint for a batch of texts at once."""
    response = client.embeddings.create(input=texts, model=model)
    # Results come back in the same order as the input texts
    return [item.embedding for item in response.data]

batch_size = 100
chunk_embeddings = []
texts_only = [chunk for source, chunk in all_chunks]

for i in range(0, len(texts_only), batch_size):
    batch = texts_only[i:i + batch_size]
    chunk_embeddings.extend(get_embeddings_batch(batch))
    print(f"Embedded {min(i + batch_size, len(texts_only))}/{len(texts_only)} chunks")

print(f"Embedding dimension: {len(chunk_embeddings[0])}")

Embedded 100/398 chunks
Embedded 200/398 chunks
Embedded 300/398 chunks
Embedded 398/398 chunks
Embedding dimension: 1536


In [9]:
# Single-text embedding helper
def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """Call the OpenAI embeddings endpoint for a single piece of text (e.g. a user query)."""
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

## Step 3: Implement Vector Search

In [7]:
# Compute cosine similarity and retrieve the top-k most relevant chunks
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity between two vectors — measures angle, not magnitude."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_top_k(query: str, k: int = 3):
    """
    Embed the query, compare it against every stored chunk embedding,
    and return the top-k most similar chunks along with their source
    document and similarity score (needed for the lab_proof.md evidence).
    """
    query_embedding = get_embedding(query)

    scored = []
    for (source, chunk), chunk_embedding in zip(all_chunks, chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_embedding)
        scored.append((score, source, chunk))

    # Sort by similarity score, descending
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

In [10]:
# Checkpoint — test with a sample query
results = retrieve_top_k("What are the requirements for human oversight of AI systems?", k=3)

for score, source, chunk in results:
    print(f"Score: {score:.4f} | Source: {source}")
    print(chunk[:200])
    print("---")

Score: 0.7094 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
to be subject to a decision based solely on automated processing when this 
produces legal effects on users or similarly significantly affects them.36 
Human oversight. Human oversight helps ensuring 
---
Score: 0.7027 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
gations, both as regards horizontally applicable rules as well as domain -specific 
regulation. 
In the following paragraphs, each requirement is explained in more detail.  
 
1.1 Human agency and ove
---
Score: 0.6715 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
es? 
 Did you take safeguards to prevent overconfidence in or overreliance on the AI system for work 
processes? 
Human oversight: 
 Did you consider the appropriate level of human control for the p
---


In [11]:
# Test with an off-topic query for comparison
results_offtopic = retrieve_top_k("What is the best recipe for lasagna?", k=3)

for score, source, chunk in results_offtopic:
    print(f"Score: {score:.4f} | Source: {source}")
    print(chunk[:150])
    print("---")

Score: 0.1295 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
lues and (3) it should be r obust, 
both from a technical and social perspective since to ensure that, even with good intentions, AI systems do not 
c
---
Score: 0.1205 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
group-artificial-intelligence).  
The reuse policy of European Commission documents is regulated by Decision 2011/8 33/EU (OJ L 330, 14. 12.2011, p.39
---
Score: 0.1183 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
and New Technologies (“EGE”) 
proposed a set of 9 basic principles, based on the fundamental values laid down in the EU Treaties and Charter. 24 
We b
---


## Step 4: Build RAG Query Function

In [12]:
# Build the full RAG pipeline — retrieve context, then generate an answer
def rag_query(question: str, k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve top-k chunks, build a grounded prompt,
    and generate an answer. Returns a trace dict capturing every step,
    so the answer can be traced back to its evidence for lab_proof.md.
    """
    top_chunks = retrieve_top_k(question, k=k)
    context = "\n\n".join(f"[{source}]\n{chunk}" for score, source, chunk in top_chunks)

    prompt = f"""Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content

    # This trace is the proof-of-grounding evidence for lab_proof.md
    trace = {
        "query": question,
        "retrieved_chunks": [
            {"source": source, "score": round(float(score), 4), "text": chunk}
            for score, source, chunk in top_chunks
        ],
        "final_answer": answer,
    }
    return trace

In [13]:
# Checkpoint — test with your grounded question
trace_success = rag_query("What are the requirements for human oversight of AI systems?")
print("ANSWER:")
print(trace_success["final_answer"])

ANSWER:
The requirements for human oversight of AI systems include:

1. Ensuring that AI systems support human autonomy and decision-making.
2. Implementing governance mechanisms such as human-in-the-loop (HITL), human-on-the-loop (HOTL), or human-in-command (HIC) approaches to allow for human intervention.
3. Considering the appropriate level of human control for the particular AI system and use case.
4. Describing the level of human control or involvement, including identifying who is the “human in control” and the moments or tools for human intervention.
5. Establishing mechanisms and measures to ensure human control or oversight.


In [14]:
# Checkpoint — test with your off-topic question (the failure case)
trace_failure = rag_query("What is the best recipe for lasagna?")
print("ANSWER:")
print(trace_failure["final_answer"])

ANSWER:
I don't know.


### Testing

In [ ]:
# Test whether the smaller (UNESCO) document can surface despite imbalance
test_queries = [
    "What is the Readiness Assessment Methodology?",
    "What is the AI Ethics Experts Without Borders network?",
    "Does the Recommendation prohibit social scoring or mass surveillance?",
]

for query in test_queries:
    print(f"QUERY: {query}")
    results = retrieve_top_k(query, k=5)
    for score, source, chunk in results:
        short_source = "UNESCO" if "eng" in source else "HLEG"
        print(f"  Score: {score:.4f} | {short_source}")
    print("---")

QUERY: What is the Readiness Assessment Methodology?
  Score: 0.4504 | UNESCO
  Score: 0.3966 | UNESCO
  Score: 0.3777 | HLEG
  Score: 0.3775 | HLEG
  Score: 0.3761 | HLEG
---
QUERY: What is the AI Ethics Experts Without Borders network?
  Score: 0.6331 | UNESCO
  Score: 0.5744 | HLEG
  Score: 0.5716 | UNESCO
  Score: 0.5621 | UNESCO
  Score: 0.5574 | UNESCO
---
QUERY: Does the Recommendation prohibit social scoring or mass surveillance?
  Score: 0.6210 | UNESCO
  Score: 0.4998 | HLEG
  Score: 0.4987 | UNESCO
  Score: 0.4764 | UNESCO
  Score: 0.4747 | UNESCO
---


In [ ]:
# Harder edge cases — overlapping themes, ambiguous phrasing, short factual answers
edge_case_queries = [
    "How many core values are there?",  # HLEG has 4 ethical principles, UNESCO has 4 core values — similar counts, different frameworks
    "How many Member States adopted the Recommendation?",  # short factual answer, only in UNESCO
    "What are the ethical principles for AI?",  # generic phrasing both docs would claim relevance to
    "When was the Recommendation adopted?",  # short factual date, only in UNESCO
]

for query in edge_case_queries:
    print(f"QUERY: {query}")
    results = retrieve_top_k(query, k=5)
    for score, source, chunk in results:
        short_source = "UNESCO" if "eng" in source else "HLEG"
        preview = chunk[:80].replace("\n", " ")
        print(f"  Score: {score:.4f} | {short_source} | {preview}...")
    print("---")

QUERY: How many core values are there?
  Score: 0.5130 | UNESCO | mendation are four core values which lay the  foundations for AI systems that wo...
  Score: 0.4235 | HLEG | ditability, minimisation and reporting of negative impact, trade-offs and redres...
  Score: 0.4220 | HLEG | , quality and integrity of data, and access to data   4 Transparency   Including...
  Score: 0.4133 | HLEG | and New Technologies (“EGE”)  proposed a set of 9 basic principles, based on the...
  Score: 0.4036 | HLEG | ity.  24  More recently, the AI4Peopl e’s taskforce has surveyed the aforementio...
---
QUERY: How many Member States adopted the Recommendation?
  Score: 0.4479 | UNESCO | f STEM career ads.  Management Science, 65(7), pp. 2966-81. 16 There is still a ...
  Score: 0.3974 | UNESCO | ly implement the Recommendation.  The RAM may address questions such as: • Are l...
  Score: 0.3969 | UNESCO | hapters  that call for better governance of data,  gender equality, and importan...
  Score: 0.3937 | U

In [18]:
# Diagnostic — find where the correct answer chunk ranks, and why it loses

def find_chunks_containing(substring: str):
    """Find all chunks containing a given substring, with their index in all_chunks."""
    matches = []
    for i, (source, chunk) in enumerate(all_chunks):
        if substring in chunk:
            matches.append((i, source, chunk))
    return matches

# Step 1: locate the chunk(s) that actually contain the answer
matches = find_chunks_containing("193 Member States")
for idx, source, chunk in matches:
    print(f"Found at index {idx} | Source: {source}")
    print(chunk)
    print("===")

Found at index 363 | Source: 385082eng.pdf
ready marginalised groups.
Recognising the urgency of this challenge, UNESCO published the first-
ever global standard on AI ethics – the ‘Recommendation on the Ethics of 
Artificial Intelligence’ (hereafter “the Recommendation”). This framework 
was adopted by all 193 Member States in November 2021. 
The protection of human rights and dignity is the cornerstone of the 
Recommendation, based on the advancement of fundamental principles 
such as transparency and fairness, always remembering the i
===


In [19]:
# Rank of the correct chunk within the FULL sorted list (not just top-5)
def full_rank(query: str, target_index: int):
    """Return the rank and score of a specific chunk (by its all_chunks index) for a given query."""
    query_embedding = get_embedding(query)
    scored = []
    for i, ((source, chunk), chunk_embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
        score = cosine_similarity(query_embedding, chunk_embedding)
        scored.append((score, i))
    scored.sort(key=lambda x: x[0], reverse=True)

    for rank, (score, i) in enumerate(scored, 1):
        if i == target_index:
            return rank, score
    return None, None

# Use the index found in Cell 15
target_idx = matches[0][0]  # adjust if multiple matches
rank, score = full_rank("How many Member States adopted the Recommendation?", target_idx)
print(f"Correct chunk rank: {rank} out of {len(all_chunks)} | Score: {score:.4f}")

Correct chunk rank: 12 out of 398 | Score: 0.3358


In [20]:
# Manually re-chunk the same passage with a bigger chunk_size and compare
# First, find where this passage sits in the ORIGINAL full document text
source_filename = matches[0][1]
full_text = docs[source_filename]

anchor_position = full_text.find("193 Member States")
wider_chunk = full_text[max(0, anchor_position - 500) : anchor_position + 500].strip()

print("Wider chunk (1000 chars around the answer):")
print(wider_chunk)
print("===")

# Embed and score this wider version against the same query
wider_embedding = get_embedding(wider_chunk)
query_embedding = get_embedding("How many Member States adopted the Recommendation?")
wider_score = cosine_similarity(query_embedding, wider_embedding)

print(f"Original chunk score: {score:.4f}")
print(f"Wider chunk score: {wider_score:.4f}")

Wider chunk (1000 chars around the answer):
ntial AI systems have to embed biases, contribute to 
climate degradation, threaten human rights and more. Such risks associated 
with AI have already begun to compound on top of existing inequalities, 
resulting in further harm to already marginalised groups.
Recognising the urgency of this challenge, UNESCO published the first-
ever global standard on AI ethics – the ‘Recommendation on the Ethics of 
Artificial Intelligence’ (hereafter “the Recommendation”). This framework 
was adopted by all 193 Member States in November 2021. 
The protection of human rights and dignity is the cornerstone of the 
Recommendation, based on the advancement of fundamental principles 
such as transparency and fairness, always remembering the importance 
of human oversight of AI systems.
However, what makes the Recom -
mendation exceptionally applicable 
are its extensive Policy Action Areas, 
which allow policymakers to translate 
the core values and principles